In [1]:
# Cell 1 — Environment setup
!pip install -q -U ultralytics sahi pycocotools

import os, platform, torch, ultralytics, sahi
from pathlib import Path

# Auto-detect Kaggle vs. lab machine
ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
DATA = Path("/tmp/VisDrone") if Path("/kaggle").exists() else ROOT / "VisDrone"
RUNS = ROOT / "runs"
DATA.mkdir(parents=True, exist_ok=True)
!df -h /tmp /kaggle/working

print("Python      :", platform.python_version())
print("Torch       :", torch.__version__, "| CUDA:", torch.version.cuda)
print("Ultralytics :", ultralytics.__version__)
print("SAHI        :", sahi.__version__)
print("GPUs        :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name} — {p.total_memory/1e9:.1f} GB")
print("ROOT        :", ROOT)

# Sanity check: confirms this Ultralytics version has YOLO26
from ultralytics import YOLO
_ = YOLO("yolo26n.pt")
print("YOLO26 weights loaded OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.9/155.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  6.9T  1.1T  87% /
/dev/loop2       20G  124K   20G   1% /kaggle/working
Python      : 3.12.13
Torch       : 2.10.0+cu128 | CUDA: 12.8
Ultralytics : 8.4.166
SAHI        : 0.12.7
GPUs        : 2
  [0] Tesla T4 — 15.6 GB
  [1] Tesla T4 — 15.6 G

In [2]:
# Cell 2 — Download VisDrone2019-DET + convert to YOLO format
# VisDrone annotation row: x,y,w,h,score,category,truncation,occlusion
#   score==0 or category==0 -> "ignored region" (saved separately for fair evaluation)
#   category==11 ("others") -> dropped, as in the official protocol
#   categories 1..10 -> YOLO classes 0..9
import json, zipfile, urllib.request
from collections import Counter
from PIL import Image
from tqdm import tqdm

BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
SPLITS = {
    "train":          "VisDrone2019-DET-train",
    "val":            "VisDrone2019-DET-val",
    "test-dev":       "VisDrone2019-DET-test-dev",
    "test-challenge": "VisDrone2019-DET-test-challenge",
}
NAMES = ["pedestrian", "people", "bicycle", "car", "van",
         "truck", "tricycle", "awning-tricycle", "bus", "motor"]

def download(name):
    z = DATA / f"{name}.zip"
    if (DATA / name).exists():
        return True
    try:
        print("Downloading", name)
        urllib.request.urlretrieve(BASE + z.name, z)
        with zipfile.ZipFile(z) as f:
            f.extractall(DATA)
        z.unlink()
        return True
    except Exception as e:
        print(f"  skipped {name}: {e}")
        return False

def convert(split_dir):
    img_dir, ann_dir = split_dir / "images", split_dir / "annotations"
    lbl_dir = split_dir / "labels"
    lbl_dir.mkdir(exist_ok=True)
    if not ann_dir.exists():
        print(f"  {split_dir.name}: no annotations (expected for test-challenge)")
        return None
    ignores, counts, sizes = {}, Counter(), []
    for ann in tqdm(sorted(ann_dir.glob("*.txt")), desc=split_dir.name):
        W, H = Image.open(img_dir / f"{ann.stem}.jpg").size
        out, ign = [], []
        for row in ann.read_text().strip().splitlines():
            v = row.strip().rstrip(",").split(",")
            if len(v) < 6:
                continue
            x, y, w, h, score, cat = map(int, v[:6])
            if w <= 0 or h <= 0:
                continue
            if score == 0 or cat == 0:
                ign.append([x, y, w, h])
                continue
            if cat == 11:
                continue
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
            bw, bh = (x2 - x1) / W, (y2 - y1) / H
            out.append(f"{cat-1} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            counts[cat - 1] += 1
            sizes.append((x2 - x1) * (y2 - y1))
        (lbl_dir / ann.name).write_text("\n".join(out))
        ignores[ann.stem] = ign
    (split_dir / "ignore_regions.json").write_text(json.dumps(ignores))
    return counts, sizes

stats = {}
for split, name in SPLITS.items():
    if download(name):
        r = convert(DATA / name)
        if r:
            stats[split] = r

# Dataset YAML (test = test-dev, since it has ground truth)
yaml_text = f"""path: {DATA}
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images
names:
""" + "\n".join(f"  {i}: {n}" for i, n in enumerate(NAMES))
(DATA / "visdrone.yaml").write_text(yaml_text)

# Stats worth keeping for the paper's dataset section
for split, (counts, sizes) in stats.items():
    n = len(sizes)
    tiny = sum(s < 16**2 for s in sizes) / n
    small = sum(s < 32**2 for s in sizes) / n
    print(f"\n{split}: {n} instances | <16px: {tiny:.1%} | <32px (COCO 'small'): {small:.1%}")
    for i, nm in enumerate(NAMES):
        print(f"  {nm:16s} {counts[i]:7d}")

VisDrone2019-DET-train: 100%|██████████| 6471/6471 [00:05<00:00, 1182.18it/s]


VisDrone2019-DET-val: 100%|██████████| 548/548 [00:00<00:00, 1881.17it/s]


VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:00<00:00, 2287.77it/s]


  VisDrone2019-DET-test-challenge: no annotations (expected for test-challenge)

train: 343204 instances | <16px: 26.0% | <32px (COCO 'small'): 60.5%
  pedestrian         79337
  people             27059
  bicycle            10480
  car               144866
  van                24956
  truck              12875
  tricycle            4812
  awning-tricycle     3246
  bus                 5926
  motor              29647

val: 38759 instances | <16px: 30.8% | <32px (COCO 'small'): 68.6%
  pedestrian          8844
  people              5125
  bicycle             1287
  car                14064
  van                 1975
  truck                750
  tricycle            1045
  awning-tricycle      532
  bus                  251
  motor               4886

test-dev: 75102 instances | <16px: 36.4% | <32px (COCO 'small'): 67.7%
  pedestrian         21006
  people              6376
  bicycle             1302
  car                28074
  van                 5771
  truck               2659
  tricycl

In [3]:
# Cell 3 — Build D1 (ignore-masked full images) and D2 (multi-scale tiles)
import cv2, json, random, shutil, numpy as np
from concurrent.futures import ProcessPoolExecutor

TRAIN   = DATA / "VisDrone2019-DET-train"
MASKED  = DATA / "train_masked"
TILED   = DATA / "train_tiles"
TILE_SIZES = [640, 960]
OVERLAP    = 0.25
MIN_VIS    = 0.6     # keep a cut box only if >=60% of it is inside the tile
EMPTY_KEEP = 0.1     # keep 10% of tiles that contain no objects
GREY = 114
for d in (MASKED, TILED):
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)

IGN = json.loads((TRAIN / "ignore_regions.json").read_text())

def load_boxes(stem, W, H):
    p = TRAIN / "labels" / f"{stem}.txt"
    rows = [r.split() for r in p.read_text().splitlines() if r.strip()] if p.exists() else []
    out = []
    for c, cx, cy, w, h in rows:
        cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
        out.append([int(c), cx-w/2, cy-h/2, cx+w/2, cy+h/2])
    return np.array(out, dtype=np.float32).reshape(-1, 5)

def starts(L, T, step):
    if L <= T:
        return [0]
    s = list(range(0, L - T + 1, step))
    if s[-1] + T < L:
        s.append(L - T)
    return s

def process(stem):
    rng = random.Random(stem)
    img = cv2.imread(str(TRAIN / "images" / f"{stem}.jpg"))
    H, W = img.shape[:2]
    for x, y, w, h in IGN.get(stem, []):
        img[max(0, y):y+h, max(0, x):x+w] = GREY
    # D1: masked full image + original labels
    cv2.imwrite(str(MASKED / "images" / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    shutil.copy(TRAIN / "labels" / f"{stem}.txt", MASKED / "labels" / f"{stem}.txt")

    boxes = load_boxes(stem, W, H)
    n = 0
    for T in TILE_SIZES:
        if T >= max(W, H):
            continue
        step = int(T * (1 - OVERLAP))
        for y0 in starts(H, T, step):
            for x0 in starts(W, T, step):
                x1, y1, x2, y2 = x0, y0, min(x0+T, W), min(y0+T, H)
                tile = img[y1:y2, x1:x2].copy()
                tw, th = x2 - x1, y2 - y1
                lines = []
                if len(boxes):
                    bx1 = np.clip(boxes[:, 1], x1, x2); by1 = np.clip(boxes[:, 2], y1, y2)
                    bx2 = np.clip(boxes[:, 3], x1, x2); by2 = np.clip(boxes[:, 4], y1, y2)
                    inter = (bx2 - bx1) * (by2 - by1)
                    area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 4] - boxes[:, 2])
                    vis = np.where(area > 0, inter / area, 0)
                    for i in np.where(inter > 0)[0]:
                        if vis[i] >= MIN_VIS:
                            cx = ((bx1[i] + bx2[i]) / 2 - x1) / tw
                            cy = ((by1[i] + by2[i]) / 2 - y1) / th
                            w_ = (bx2[i] - bx1[i]) / tw
                            h_ = (by2[i] - by1[i]) / th
                            lines.append(f"{int(boxes[i,0])} {cx:.6f} {cy:.6f} {w_:.6f} {h_:.6f}")
                        else:
                            # grey out badly-cut objects so they aren't unlabeled positives
                            tile[int(by1[i]-y1):int(by2[i]-y1), int(bx1[i]-x1):int(bx2[i]-x1)] = GREY
                if not lines and rng.random() > EMPTY_KEEP:
                    continue
                name = f"{stem}_t{T}_{x0}_{y0}"
                cv2.imwrite(str(TILED / "images" / f"{name}.jpg"), tile, [cv2.IMWRITE_JPEG_QUALITY, 95])
                (TILED / "labels" / f"{name}.txt").write_text("\n".join(lines))
                n += 1
    return n

stems = [p.stem for p in (TRAIN / "images").glob("*.jpg")]
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    counts = list(tqdm(ex.map(process, stems, chunksize=16), total=len(stems), desc="masking + tiling"))
print(f"D1: {len(stems)} masked images | D2 tiles: {sum(counts)}")

VAL = "VisDrone2019-DET-val/images"
TEST = "VisDrone2019-DET-test-dev/images"
names_block = "names:\n" + "\n".join(f"  {i}: {n}" for i, n in enumerate(NAMES))
(DATA / "d1_full.yaml").write_text(
    f"path: {DATA}\ntrain: train_masked/images\nval: {VAL}\ntest: {TEST}\n{names_block}")
(DATA / "d2_tiles.yaml").write_text(
    f"path: {DATA}\ntrain:\n  - train_masked/images\n  - train_tiles/images\nval: {VAL}\ntest: {TEST}\n{names_block}")
!du -sh {MASKED} {TILED}

masking + tiling: 100%|██████████| 6471/6471 [03:08<00:00, 34.36it/s]

D1: 6471 masked images | D2 tiles: 63745
2.8G	/tmp/VisDrone/train_masked


12G	/tmp/VisDrone/train_tiles


In [4]:
# Cell 5 — K1: YOLO26m on D2 (masked full images + tiles), both T4s.
# time=10.5 fits the run into one Kaggle session: Ultralytics sets the epoch
# count and learning-rate schedule to finish in 10.5 h (this overrides epochs).
!yolo detect train model=yolo26m.pt data={DATA}/d2_tiles.yaml imgsz=1024 batch=16 \
    device=0,1 workers=4 time=10.5 epochs=20 patience=100 cos_lr=True close_mosaic=3 \
    mosaic=1.0 mixup=0.1 scale=0.5 fliplr=0.5 max_det=1000 \
    project={RUNS} name=K1_y26m_tiles exist_ok=True plots=True seed=0

!ls -lh {RUNS}/K1_y26m_tiles/weights

Ultralytics 8.4.166 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=3, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/VisDrone/d2_tiles.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=1000, mixup=0.1, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0